# Tutorial 1: Individual Star Models

This tutorial explores individual star modeling using EEPTracks and StarEvolTrack.

## Topics Covered

1. **EEPTracks** for parameter prediction along evolutionary tracks
2. **StarEvolTrack** for on-the-fly SED generation  
3. **Exploring parameter space** (mass, metallicity, age)
4. **Binary star modeling**
5. **Extinction and distance effects**

## Prerequisites

This tutorial requires the following brutus data files:
- `MIST_1.2_EEPtrk.h5` - MIST evolutionary tracks
- `nn_c3k.h5` - Neural network for bolometric corrections

If you don't have these files, run the optional download cell below.

In [ ]:
# Optional: Download required data files (only run if needed)
# Uncomment the lines below to download

from brutus.data import fetch_isos
fetch_isos(target_dir='../data/DATAFILES/')  # Downloads tracks, isochrones, and neural networks

In [ ]:
# Imports and setup
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Import tutorial utilities
from tutorial_utils import (
    set_plot_style, 
    find_brutus_data_file, 
    save_figure as save_fig_util,
    print_section
)

# Set plot style
set_plot_style()
plt.rcParams['figure.figsize'] = (10, 6)

# Create plots directory if needed
plots_dir = Path('plots/tutorial_01')
plots_dir.mkdir(parents=True, exist_ok=True)

def save_figure(fig, name):
    """Helper to save figures."""
    filepath = plots_dir / f"{name}.png"
    fig.savefig(filepath, dpi=150, bbox_inches='tight')
    print(f"  Saved: {filepath}")

## Section 1: EEPTracks - Parameter Prediction

EEPTracks provides stellar parameter predictions along evolutionary tracks.
It interpolates MIST stellar evolution models to predict physical parameters
at any point along a star's evolution.

### Key Concepts

- **EEP (Equivalent Evolutionary Point)**: A normalized coordinate along stellar evolution tracks
- **Tracks vs Isochrones**: Tracks follow individual stars, isochrones are snapshots of populations
- **Parameter prediction**: Get stellar properties (Teff, log g, luminosity) at any EEP

In [ ]:
from brutus.core import EEPTracks
from brutus.data import filters

# Initialize EEPTracks
print("Loading MIST evolutionary tracks...")
mistfile = find_brutus_data_file("MIST_1.2_EEPtrk.h5")

tracks = EEPTracks(mistfile=mistfile, verbose=False)

# Explore the parameter space covered
masses = tracks.xgrid[0]  # Initial masses
metallicities = tracks.xgrid[2]  # [Fe/H] values

print(f"✓ Loaded tracks covering {len(masses)} mass points")
print(f"  Mass range: {masses.min():.2f} - {masses.max():.2f} M☉")
print(f"  Metallicity range: {metallicities.min():.2f} - {metallicities.max():.2f}")
print(f"  Available predictions: {tracks.predictions}")

In [ ]:
# Predict parameters for a solar-mass star at different evolutionary stages
print("\nPredicting parameters for a 1 M☉ star at different evolutionary stages:\n")

# EEP ranges for different phases
eep_examples = [
    (250, "Pre-Main Sequence"),
    (350, "Zero-Age Main Sequence"),
    (400, "Middle Main Sequence"),
    (450, "Terminal-Age Main Sequence"),
    (500, "Subgiant Branch"),
    (650, "Red Giant Branch")
]

for eep, phase in eep_examples:
    try:
        # get_predictions takes [mini, eep, feh, afe]
        params = tracks.get_predictions([1.0, eep, 0.0, 0.0])
        
        # Extract specific parameters (indices based on tracks.predictions)
        loga_idx = tracks.predictions.index("loga")
        logl_idx = tracks.predictions.index("logl")
        logt_idx = tracks.predictions.index("logt")
        logg_idx = tracks.predictions.index("logg")
        
        age_gyr = 10**params[loga_idx] / 1e9
        luminosity = 10**params[logl_idx]
        teff = 10**params[logt_idx]
        logg = params[logg_idx]
        
        print(f"EEP {eep:3d} ({phase:25s}): Age={age_gyr:5.2f} Gyr, L={luminosity:6.2f} L☉, Teff={teff:5.0f} K, log g={logg:4.2f}")
    except:
        print(f"EEP {eep:3d} ({phase:25s}): Not available for 1 M☉ star")

## Section 2: StarEvolTrack - SED Generation

StarEvolTrack generates SEDs using neural networks for bolometric corrections.
This provides on-the-fly photometry generation at any point in parameter space.

### Key Features

- Fast SED generation using neural networks
- Support for any photometric filter system
- Binary star modeling capabilities
- Extinction and distance effects

In [ ]:
from brutus.core import StarEvolTrack

# Set up filters (Pan-STARRS + 2MASS)
filt = filters.ps[:-2] + filters.tmass  
print(f"Using filters: {', '.join(filt)}")

# Initialize StarEvolTrack
nnfile = find_brutus_data_file("nn_c3k.h5")
star = StarEvolTrack(tracks=tracks, nnfile=nnfile, filters=filt, verbose=False)

print("✓ StarEvolTrack initialized with neural network bolometric corrections")

In [ ]:
# Generate SED for a solar-like star
print("\nGenerating SED for solar-like star (1 M☉, solar metallicity, MS):")

# Generate magnitudes at 10 pc
mags, params, _ = star.get_seds(mini=1.0, feh=0.0, eep=350, dist=10.0)

print(f"\nMagnitudes at 10 pc:")
for i, (f, m) in enumerate(zip(filt, mags)):
    print(f"  {f:10s}: {m:6.3f} mag")

# Calculate some colors
g_idx = filters.ps.index('PS_g')
r_idx = filters.ps.index('PS_r')
i_idx = filters.ps.index('PS_i')

print(f"\nColors:")
print(f"  g - r = {mags[g_idx] - mags[r_idx]:.3f}")
print(f"  r - i = {mags[r_idx] - mags[i_idx]:.3f}")

In [ ]:
# Plot the SED and main sequence tracks
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# Panel 1: SED in magnitudes
ax1.plot(range(len(filt)), mags, 'o-', color='orange', lw=2, ms=8)
ax1.set_xticks(range(len(filt)))
ax1.set_xticklabels(filt, rotation=45, ha='right')
ax1.set_ylabel('Magnitude (at 10 pc)')
ax1.set_title('Solar-like Star SED')
ax1.grid(True, alpha=0.3)
ax1.invert_yaxis()

# Panel 2: Color-magnitude diagram with tracks
colors = ['blue', 'green', 'orange', 'red']
masses = [0.5, 0.8, 1.0, 1.5]

for mass, color in zip(masses, colors):
    eep_range = np.linspace(202, 454, 100)  # Main sequence only
    mags_arr = []
    
    for eep in eep_range:
        try:
            m, _, _ = star.get_seds(mini=mass, feh=0.0, eep=eep, dist=10.0)
            mags_arr.append(m)
        except:
            continue
    
    if mags_arr:
        mags_arr = np.array(mags_arr)
        g_idx = filters.ps.index('PS_g')
        r_idx = filters.ps.index('PS_r')
        
        ax2.plot(mags_arr[:, g_idx] - mags_arr[:, r_idx], 
                 mags_arr[:, g_idx],
                 color=color, lw=2, alpha=0.7, label=f'{mass} M☉')

ax2.set_xlabel('g - r')
ax2.set_ylabel('g magnitude')
ax2.set_title('Main Sequence Tracks')
ax2.invert_yaxis()
ax2.legend(loc='upper left')
ax2.grid(True, alpha=0.3)

plt.suptitle('StarEvolTrack SED Generation', fontsize=14, fontweight='bold')
plt.tight_layout()
save_figure(fig, 'sed_generation')
plt.show()

## Section 3: Stellar Evolution Along Tracks

Let's explore how stellar parameters evolve along evolutionary tracks,
from pre-main sequence through the giant branch.

In [ ]:
# Use Gaia filters for the HRD
filt_gaia = filters.gaia
star_gaia = StarEvolTrack(tracks=tracks, nnfile=nnfile, filters=filt_gaia, verbose=False)

# Define EEP ranges for different evolutionary phases
eep_phases = {
    "Pre-MS": (202, 353),
    "MS": (353, 454),
    "SGB": (454, 605),
    "RGB": (605, 707),
    "HB/AGB": (707, 808),
}

print("EEP ranges for evolutionary phases:")
for phase, (eep_min, eep_max) in eep_phases.items():
    print(f"  {phase:8s}: EEP {eep_min:3d} - {eep_max:3d}")

In [ ]:
# Create comprehensive evolution plots
fig = plt.figure(figsize=(15, 10))

# Set up subplots
ax1 = plt.subplot(2, 3, 1)  # HRD
ax2 = plt.subplot(2, 3, 2)  # CMD
ax3 = plt.subplot(2, 3, 3)  # Age evolution
ax4 = plt.subplot(2, 3, 4)  # Age in Gyr
ax5 = plt.subplot(2, 3, 5)  # Kiel diagram
ax6 = plt.subplot(2, 3, 6)  # EEP phases

# Generate tracks for different masses
masses = [0.5, 1.0, 2.0, 5.0]
colors = ['purple', 'blue', 'green', 'red']

for mass, color in zip(masses, colors):
    eep_grid = np.linspace(202, 808, 500)
    
    # Collect parameters along track
    params_list = []
    mags_list = []
    
    for eep in eep_grid:
        try:
            params_arr = tracks.get_predictions([mass, eep, 0.0, 0.0])
            params = {label: params_arr[i] for i, label in enumerate(tracks.predictions)}
            mags, _, _ = star_gaia.get_seds(mini=mass, feh=0.0, eep=eep, dist=1000.0)  # at 1 kpc
            params_list.append(params)
            mags_list.append(mags)
        except:
            continue
    
    if not params_list:
        continue
    
    # Convert to arrays
    params_arr = {key: np.array([p[key] for p in params_list]) for key in params_list[0].keys()}
    mags_arr = np.array(mags_list)
    
    # Panel 1: HRD
    ax1.plot(params_arr['logt'], params_arr['logl'], 
             color=color, lw=2, alpha=0.7, label=f'{mass} M☉')
    
    # Panel 2: Gaia CMD
    bp_rp = mags_arr[:, 1] - mags_arr[:, 2]  # BP - RP
    g_mag = mags_arr[:, 0]  # G
    ax2.plot(bp_rp, g_mag, color=color, lw=2, alpha=0.7)
    
    # Panel 3: Age evolution
    ax3.plot(eep_grid[:len(params_list)], params_arr['loga'], 
             color=color, lw=2, alpha=0.7)
    
    # Panel 4: Age in Gyr
    ages = 10**params_arr['loga'] / 1e9  # Convert to Gyr
    ax4.plot(eep_grid[:len(params_list)], ages, 
             color=color, lw=2, alpha=0.7)
    
    # Panel 5: Kiel diagram
    ax5.plot(params_arr['logt'], params_arr['logg'], 
             color=color, lw=2, alpha=0.7)

# Panel 6: Show evolutionary phases
phase_colors = ['yellow', 'orange', 'red', 'darkred', 'purple']
y_pos = 0.8
for (phase, (eep_min, eep_max)), color in zip(eep_phases.items(), phase_colors):
    ax6.barh(y_pos, eep_max - eep_min, left=eep_min, height=0.15,
             color=color, alpha=0.6, label=phase)
    y_pos -= 0.2

# Format all plots
ax1.set_xlabel('log T_eff (K)')
ax1.set_ylabel('log L/L☉')
ax1.set_title('Hertzsprung-Russell Diagram')
ax1.invert_xaxis()
ax1.legend(fontsize=8)
ax1.grid(True, alpha=0.3)

ax2.set_xlabel('BP - RP')
ax2.set_ylabel('G magnitude')
ax2.set_title('Gaia CMD')
ax2.invert_yaxis()
ax2.set_ylim(15, -5)
ax2.grid(True, alpha=0.3)

ax3.set_xlabel('EEP')
ax3.set_ylabel('log(Age/yr)')
ax3.set_title('Age Evolution')
ax3.grid(True, alpha=0.3)

ax4.set_xlabel('EEP')
ax4.set_ylabel('Age [Gyr]')
ax4.set_title('Age vs EEP')
ax4.set_ylim(0, 15)
ax4.grid(True, alpha=0.3)

ax5.set_xlabel('log T_eff (K)')
ax5.set_ylabel('log g (cgs)')
ax5.set_title('Kiel Diagram')
ax5.invert_xaxis()
ax5.invert_yaxis()
ax5.grid(True, alpha=0.3)

ax6.set_xlabel('EEP')
ax6.set_ylabel('Evolutionary Phase')
ax6.set_title('EEP Phase Mapping')
ax6.set_xlim(200, 810)
ax6.set_ylim(0, 1)
ax6.legend(fontsize=8, loc='center')
ax6.set_yticks([])

plt.suptitle('Stellar Evolution Along MIST Tracks', fontsize=16, fontweight='bold')
plt.tight_layout()
save_figure(fig, 'stellar_evolution')
plt.show()

print("✓ Generated stellar evolution plots showing:")
print("  - HRD evolution for different masses")
print("  - Position in Gaia CMD")
print("  - Age evolution")
print("  - Mapping of EEP to evolutionary phases")

## Section 4: Metallicity Effects

Metallicity significantly affects stellar evolution and photometry.
Let's explore how [Fe/H] changes stellar properties and colors.

In [ ]:
# Create metallicity comparison plots
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

# Different metallicities to explore
feh_values = [-2.0, -1.0, -0.5, 0.0, 0.3]
colors = plt.cm.coolwarm(np.linspace(0, 1, len(feh_values)))

# Fixed mass for comparison
test_mass = 1.0
ms_eeps = np.linspace(350, 450, 50)  # Main sequence only

for feh, color in zip(feh_values, colors):
    # Collect data
    params_list = []
    mags_list = []
    
    for eep in ms_eeps:
        try:
            params_arr = tracks.get_predictions([test_mass, eep, feh, 0.0])
            params = {label: params_arr[i] for i, label in enumerate(tracks.predictions)}
            mags, _, _ = star.get_seds(mini=test_mass, feh=feh, eep=eep, dist=10.0)
            params_list.append(params)
            mags_list.append(mags)
        except:
            continue
    
    if not params_list:
        continue
    
    params_arr = {key: np.array([p[key] for p in params_list]) for key in params_list[0].keys()}
    mags_arr = np.array(mags_list)
    
    # Panel 1: HRD
    axes[0, 0].plot(params_arr['logt'], params_arr['logl'],
                    color=color, lw=2, alpha=0.8, label=f'[Fe/H] = {feh:.1f}')
    
    # Panel 2: Optical CMD
    g_idx = filters.ps.index('PS_g')
    r_idx = filters.ps.index('PS_r')
    axes[0, 1].plot(mags_arr[:, g_idx] - mags_arr[:, r_idx], mags_arr[:, g_idx],
                    color=color, lw=2, alpha=0.8)
    
    # Panel 3: NIR CMD
    j_idx = len(filters.ps[:-2]) + filters.tmass.index('2MASS_J')
    k_idx = len(filters.ps[:-2]) + filters.tmass.index('2MASS_Ks')
    axes[0, 2].plot(mags_arr[:, j_idx] - mags_arr[:, k_idx], mags_arr[:, j_idx],
                    color=color, lw=2, alpha=0.8)
    
    # Panel 4: Age at turnoff
    turnoff_idx = np.argmax(params_arr['logl'])
    turnoff_age = 10**(params_arr['loga'][turnoff_idx]) / 1e9  # Gyr
    axes[1, 0].scatter(feh, turnoff_age, color=color, s=100, zorder=5)
    
    # Panel 5: Temperature vs metallicity
    mean_teff = np.mean(10**params_arr['logt'])
    axes[1, 1].scatter(feh, mean_teff, color=color, s=100, zorder=5)
    
    # Panel 6: Color vs metallicity
    mean_color = np.mean(mags_arr[:, g_idx] - mags_arr[:, r_idx])
    axes[1, 2].scatter(feh, mean_color, color=color, s=100, zorder=5)

# Format plots
axes[0, 0].set_xlabel('log T_eff')
axes[0, 0].set_ylabel('log L/L☉')
axes[0, 0].set_title('Main Sequence HRD')
axes[0, 0].invert_xaxis()
axes[0, 0].legend(fontsize=8)
axes[0, 0].grid(True, alpha=0.3)

axes[0, 1].set_xlabel('g - r')
axes[0, 1].set_ylabel('g magnitude')
axes[0, 1].set_title('Optical CMD')
axes[0, 1].invert_yaxis()
axes[0, 1].grid(True, alpha=0.3)

axes[0, 2].set_xlabel('J - Ks')
axes[0, 2].set_ylabel('J magnitude')
axes[0, 2].set_title('Near-IR CMD')
axes[0, 2].invert_yaxis()
axes[0, 2].grid(True, alpha=0.3)

axes[1, 0].set_xlabel('[Fe/H]')
axes[1, 0].set_ylabel('MS Turnoff Age (Gyr)')
axes[1, 0].set_title('Age-Metallicity Relation')
axes[1, 0].grid(True, alpha=0.3)

axes[1, 1].set_xlabel('[Fe/H]')
axes[1, 1].set_ylabel('Mean MS T_eff (K)')
axes[1, 1].set_title('Temperature-Metallicity')
axes[1, 1].grid(True, alpha=0.3)

axes[1, 2].set_xlabel('[Fe/H]')
axes[1, 2].set_ylabel('Mean MS (g - r)')
axes[1, 2].set_title('Color-Metallicity')
axes[1, 2].grid(True, alpha=0.3)

plt.suptitle('Metallicity Effects on Stellar Properties', fontsize=16, fontweight='bold')
plt.tight_layout()
save_figure(fig, 'metallicity_effects')
plt.show()

print("✓ Metallicity effects demonstrated:")
print("  - Metal-poor stars are bluer and hotter")
print("  - MS turnoff age depends on metallicity")
print("  - Color-metallicity relations for calibration")

## Section 5: Binary Star Modeling

Unresolved binaries significantly affect observed photometry.
StarEvolTrack can model binary systems using the secondary mass fraction (SMF).

### Binary Parameters

- **SMF (Secondary Mass Fraction)**: q = M₂/M₁ where M₁ is the primary mass
- **Equal-age assumption**: Both stars have the same age and metallicity
- **Combined light**: Total flux is the sum of both components

In [ ]:
# Set up for binary modeling
filt_binary = filters.gaia + filters.ps[:3]  # Gaia + PS optical
star_binary = StarEvolTrack(tracks=tracks, nnfile=nnfile, filters=filt_binary, verbose=False)

# Binary parameters to explore
primary_mass = 1.0  # Solar mass primary
smf_values = [0.0, 0.3, 0.5, 0.7, 1.0]  # Single to equal-mass binary
colors_smf = ['blue', 'cyan', 'green', 'orange', 'red']

print("Binary mass ratios to explore:")
for smf in smf_values:
    secondary_mass = primary_mass * smf
    print(f"  q = {smf:.1f}: M₁ = {primary_mass:.1f} M☉, M₂ = {secondary_mass:.1f} M☉")

In [ ]:
# Create binary sequence plots
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

# Explore different evolutionary stages
eep_stages = [(350, 'Early MS'), (400, 'Mid MS'), (450, 'Late MS')]

for panel_idx, (eep, stage) in enumerate(eep_stages):
    ax = axes[0, panel_idx]
    
    for smf, color in zip(smf_values, colors_smf):
        # Generate binary photometry
        mags, params1, params2 = star_binary.get_seds(
            mini=primary_mass, feh=0.0, eep=eep, smf=smf, dist=100.0
        )
        
        # Gaia colors
        bp_rp = mags[1] - mags[2]  # BP - RP
        g_mag = mags[0]  # G
        
        ax.scatter(bp_rp, g_mag, color=color, s=100, alpha=0.8,
                   label=f'q = {smf:.1f}' if panel_idx == 0 else None)
    
    ax.set_xlabel('BP - RP')
    ax.set_ylabel('G magnitude')
    ax.set_title(f'{stage} (EEP={eep})')
    ax.invert_yaxis()
    ax.grid(True, alpha=0.3)
    if panel_idx == 0:
        ax.legend(fontsize=8)

# Panel 4: Binary main sequence
ax = axes[1, 0]
eep_range = np.linspace(300, 454, 50)

for smf, color in zip([0.0, 0.5, 1.0], ['blue', 'orange', 'red']):
    g_mags = []
    bp_rp_colors = []
    
    for eep in eep_range:
        try:
            mags, _, _ = star_binary.get_seds(
                mini=1.0, feh=0.0, eep=eep, smf=smf, dist=100.0
            )
            g_mags.append(mags[0])
            bp_rp_colors.append(mags[1] - mags[2])
        except:
            continue
    
    if g_mags:
        ax.plot(bp_rp_colors, g_mags, color=color, lw=2, 
                alpha=0.8, label=f'q = {smf:.1f}')

ax.set_xlabel('BP - RP')
ax.set_ylabel('G magnitude')
ax.set_title('Binary Main Sequence')
ax.invert_yaxis()
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# Panel 5: Mass ratio distributions
ax = axes[1, 1]
smf_dist = np.linspace(0, 1, 100)

# Typical binary mass ratio distributions
flat_dist = np.ones_like(smf_dist)
twin_dist = np.exp(-((smf_dist - 1.0)**2) / 0.01)  # Twin peak
power_dist = smf_dist**(-0.5)  # Power law

ax.plot(smf_dist, flat_dist/flat_dist.max(), 'b-', lw=2, label='Flat')
ax.plot(smf_dist, twin_dist/twin_dist.max(), 'r-', lw=2, label='Twin excess')
ax.plot(smf_dist, power_dist/power_dist.max(), 'g-', lw=2, label='Power law')
ax.set_xlabel('Secondary Mass Fraction (q)')
ax.set_ylabel('Normalized Probability')
ax.set_title('Mass Ratio Distributions')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# Panel 6: Detection limits
ax = axes[1, 2]

# Calculate magnitude differences for detection
primary_mags, _, _ = star_binary.get_seds(mini=1.0, feh=0.0, eep=400, smf=0.0, dist=100.0)
smf_test = np.linspace(0.1, 1.0, 20)
delta_mags = []

for smf in smf_test:
    binary_mags, _, _ = star_binary.get_seds(
        mini=1.0, feh=0.0, eep=400, smf=smf, dist=100.0
    )
    delta_mag = binary_mags[0] - primary_mags[0]
    delta_mags.append(delta_mag)

ax.plot(smf_test, delta_mags, 'ko-', lw=2)
ax.axhline(0.75, color='red', ls='--', alpha=0.5, label='Typical detection limit')
ax.set_xlabel('Secondary Mass Fraction (q)')
ax.set_ylabel('ΔG (mag)')
ax.set_title('Detection Limits')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

plt.suptitle('Binary Star Modeling', fontsize=16, fontweight='bold')
plt.tight_layout()
save_figure(fig, 'binary_modeling')
plt.show()

print("✓ Binary modeling demonstrated:")
print("  - Binaries shift stars above the MS")
print("  - Effect depends on mass ratio (q)")
print("  - Detection limits for unresolved binaries")

## Section 6: Extinction and Distance Effects

Interstellar extinction and distance are critical for interpreting photometry.
Let's explore how these affect observed SEDs and colors.

### Key Parameters

- **A(V)**: Visual extinction in magnitudes
- **R(V)**: Total-to-selective extinction ratio (typically 3.1)
- **Distance modulus**: μ = 5 log₁₀(d/10) where d is in parsecs

In [ ]:
# Create extinction/distance plots
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

# Test star parameters
mini, feh, eep = 1.0, 0.0, 400

# Panel 1: Extinction effects on SED
ax = axes[0, 0]
av_values = [0.0, 0.5, 1.0, 2.0, 3.0]
colors_av = plt.cm.YlOrRd(np.linspace(0.2, 0.9, len(av_values)))

for av, color in zip(av_values, colors_av):
    mags, _, _ = star.get_seds(
        mini=mini, feh=feh, eep=eep, av=av, rv=3.1, dist=100.0
    )
    ax.plot(range(len(filt)), mags, 'o-', color=color, 
            alpha=0.8, label=f'A(V) = {av}')

ax.set_xticks(range(len(filt)))
ax.set_xticklabels(filt, rotation=45, ha='right')
ax.set_ylabel('Magnitude')
ax.set_title('Extinction Effects on SED')
ax.legend(fontsize=8)
ax.invert_yaxis()
ax.grid(True, alpha=0.3)

# Panel 2: Reddening vector in CMD
ax = axes[0, 1]
g_idx = filters.ps.index('PS_g')
r_idx = filters.ps.index('PS_r')

av_range = np.linspace(0, 3, 30)
g_mags, colors = [], []

for av in av_range:
    mags, _, _ = star.get_seds(
        mini=mini, feh=feh, eep=eep, av=av, rv=3.1, dist=100.0
    )
    g_mags.append(mags[g_idx])
    colors.append(mags[g_idx] - mags[r_idx])

scatter = ax.scatter(colors, g_mags, c=av_range, cmap='YlOrRd', s=50)
plt.colorbar(scatter, ax=ax, label='A(V)')
ax.set_xlabel('g - r')
ax.set_ylabel('g magnitude')
ax.set_title('Reddening Vector')
ax.invert_yaxis()
ax.grid(True, alpha=0.3)

# Panel 3: R(V) variations
ax = axes[0, 2]
rv_values = [2.0, 3.1, 4.0, 5.0]
colors_rv = ['blue', 'green', 'orange', 'red']

for rv, color in zip(rv_values, colors_rv):
    colors_temp = []
    for av in av_range:
        mags, _, _ = star.get_seds(
            mini=mini, feh=feh, eep=eep, av=av, rv=rv, dist=100.0
        )
        colors_temp.append(mags[g_idx] - mags[r_idx])
    ax.plot(av_range, colors_temp, color=color, lw=2, 
            alpha=0.8, label=f'R(V) = {rv}')

ax.set_xlabel('A(V)')
ax.set_ylabel('g - r color excess')
ax.set_title('R(V) Variations')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# Panel 4: Distance effects
ax = axes[1, 0]
distances = [10, 100, 500, 1000, 5000, 10000]  # pc

for i, dist in enumerate(distances):
    mags, _, _ = star.get_seds(
        mini=mini, feh=feh, eep=eep, av=0.0, rv=3.1, dist=dist
    )
    ax.scatter(i, mags[g_idx], s=100, label=f'{dist} pc' if i < 4 else None)

ax.set_xticks(range(len(distances)))
ax.set_xticklabels(distances)
ax.set_xlabel('Distance (pc)')
ax.set_ylabel('Apparent g magnitude')
ax.set_title('Distance Effects')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# Panel 5: Distance modulus
ax = axes[1, 1]
dist_range = np.logspace(1, 4, 50)  # 10 pc to 10 kpc
distance_modulus = 5 * np.log10(dist_range) - 5

ax.plot(dist_range, distance_modulus, 'k-', lw=2)
ax.axhline(0, color='gray', ls='--', alpha=0.5)
ax.axhline(5, color='gray', ls='--', alpha=0.5)
ax.axhline(10, color='gray', ls='--', alpha=0.5)
ax.set_xlabel('Distance (pc)')
ax.set_ylabel('Distance Modulus (mag)')
ax.set_title('Distance Modulus')
ax.set_xscale('log')
ax.grid(True, alpha=0.3)

# Panel 6: Combined effects
ax = axes[1, 2]
dist_grid = [100, 500, 1000, 2000]
av_grid = [0, 0.5, 1.0, 2.0]

for dist in dist_grid:
    g_mags_line = []
    colors_line = []
    for av in av_grid:
        mags, _, _ = star.get_seds(
            mini=mini, feh=feh, eep=eep, av=av, rv=3.1, dist=dist
        )
        g_mags_line.append(mags[g_idx])
        colors_line.append(mags[g_idx] - mags[r_idx])
    
    ax.plot(colors_line, g_mags_line, 'o-', 
            label=f'{dist} pc', lw=2, alpha=0.8)

ax.set_xlabel('g - r')
ax.set_ylabel('g magnitude')
ax.set_title('Distance + Extinction')
ax.invert_yaxis()
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

plt.suptitle('Extinction and Distance Effects', fontsize=16, fontweight='bold')
plt.tight_layout()
save_figure(fig, 'extinction_distance')
plt.show()

print("✓ Extinction and distance effects shown:")
print("  - Extinction reddens and dims stars")
print("  - R(V) controls extinction curve shape")
print("  - Distance modulus relationship")
print("  - Degeneracies between distance and extinction")

## Summary and Key Takeaways

This tutorial has covered the fundamental components for modeling individual stars in brutus:

### Key Classes

1. **EEPTracks**: Provides stellar parameter predictions along evolutionary tracks
   - Interpolates MIST stellar evolution models
   - Returns physical parameters (Teff, log g, luminosity, age)
   - Covers full evolutionary phases from pre-MS to post-AGB

2. **StarEvolTrack**: Generates SEDs using neural networks
   - Fast bolometric corrections via neural networks
   - Supports any photometric filter system
   - Includes binary star modeling (SMF parameter)
   - Handles extinction (A(V), R(V)) and distance

### Physical Effects

- **Metallicity**: Affects temperature, color, and evolutionary timescales
- **Binaries**: Create sequences above the main sequence
- **Extinction**: Reddens and dims stellar light
- **Distance**: Determines apparent magnitude via distance modulus

### Next Steps

- **Tutorial 2**: Stellar Population Models (Isochrones and StellarPop)
- **Tutorial 3**: Grid Generation and Performance Optimization
- **Tutorial 4**: Galactic Priors and Population Synthesis
- **Tutorial 5**: Fitting Individual Sources with BruteForce

In [ ]:
print("Tutorial 1 Complete!")
print("="*60)
print("\nGenerated plots:")
for plot_file in sorted(plots_dir.glob('*.png')):
    print(f"  - {plot_file.name}")